### "Statistics and Data Analysis" 18: Bootstrapping and Monte Carlo Sampling Statistics
In this lecture, Dr. Brunton estimates the error of our parameter estimate from the method of moments using Monte Carlo sampling and the bootstrap. He investigates a Poisson estimation problem.

https://www.youtube.com/watch?v=wsU7YLcPXmE&list=PLMrJAkhIeNNT14qn1c5qdL29A1UaHamjx&index=18

In [ ]:
# Alpha particles emmitted by Americium 241
# Example from Rice, taken from Berkson 1966
# 10220 Gieger clikcs (emissions) were measured, 
# specifically time between clicks data is binned into 10-second intervals
# Clicks per 10-second interval should be Poisson (time between is exponential)

import numpy as np
import matplotlib.pyplot as plt

# 1. Raw Data from Americium 241 experiment (counts in 10s intervals)
click_tenseconds = np.arange(0, 18) # Number of counts per 10-second interval
observed_counts = np.array([1, 6, 11, 28, 56, 105, 126, 146, 164, 161, 123, 101, 74, 53, 23, 15, 9, 5])

# 2. Plotting
plt.figure(figsize=(10, 6))

# Plotting the observed data
plt.plot(click_tenseconds, observed_counts,  'o-',color='gray', linewidth=2, label='Observed Data (Am-241)')

plt.title('Observed Alpha Particle Emission per 10-Second Interval', fontsize=14)
plt.xlabel('Number of Click per 10s Interval', fontsize=12)
plt.ylabel('Observed Count', fontsize=12)
plt.legend()
plt.grid(True)
plt.xticks(click_tenseconds)
plt.show()

In [ ]:
from scipy.stats import poisson

# Calculate the sample mean
sample_mean = np.average(click_tenseconds, weights=observed_counts)

total_count = np.sum(observed_counts)

# Fit a Poisson distribution using the sample mean as lambda
lambda_hat = sample_mean
poisson_dist = poisson.pmf(click_tenseconds, mu=lambda_hat)* total_count

# Plotting
plt.figure(figsize=(10, 6))
# Plotting the observed data
plt.plot(click_tenseconds, observed_counts, 'o-', linewidth=2, color='gray', label='Observed Data (Am-241)')

# Plot the theoretical fit as a line with markers
plt.plot(click_tenseconds, poisson_dist, 'x--', color='red', linewidth=2, label=f'Poisson Fit ($\\hat{{\\lambda}}$={lambda_hat:.2f})')

plt.title('Fitting a Poisson Distribution to Radioactive Decay Data', fontsize=14)
plt.xlabel('Number of Click per 10s Interval', fontsize=12)
plt.ylabel('Observed Count', fontsize=12)
plt.legend()
plt.grid(True)

plt.xticks(click_tenseconds)
plt.show()

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of bootstrap samples
num_bootstraps = 1000
bootstrap_lambdas = []

# Bootstrap procedure
for _ in range(num_bootstraps):
    # Generate a synthetic dataset using the estimated lambda
    synthetic_data = np.random.poisson(lam=lambda_hat, size=sum(observed_counts))
    
    # Bin the synthetic data to match the observed format (number of counts per 10-second interval)
    sythetic_hist, _ = np.histogram(synthetic_data, bins=np.arange(0, 19))
    
    # Calculate the new lambda estimate from the synthetic data
    new_lambda = np.average(click_tenseconds, weights=sythetic_hist)
    
    # Store the new lambda estimate
    bootstrap_lambdas.append(new_lambda)
    
# Plot the distribution of bootstrap lambda estimates
plt.figure(figsize=(10, 6))
plt.hist(bootstrap_lambdas, bins=30, color='green', edgecolor='black', alpha=0.7)
plt.title('Bootstrap Distribution of Lambda Estimates', fontsize=14)
plt.xlabel('Lambda Estimate', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True)

# Print the mean and confidence interval of the bootstrap lambda estimates
bootstrap_mean = np.mean(bootstrap_lambdas)
bootstrap_std = np.std(bootstrap_lambdas)
bootstrap_confidence_interval = (bootstrap_mean - 1.96 * bootstrap_std, bootstrap_mean + 1.96 * bootstrap_std)
print(f"Bootstrap Mean of Lambda: {bootstrap_mean:.2f}")
print(f"Bootstrap Std Dev of Lambda: {bootstrap_std:.2f}")
print(f"Bootstrap Confidence Interval: ({bootstrap_confidence_interval[0]:.2f}, {bootstrap_confidence_interval[1]:.2f})")
    

In [ ]:
from scipy.stats import norm

# Set random seed for reproducibility
np.random.seed(42)

# Number of bootstrap samples
num_bootstraps = 1000
bootstrap_lambdas = []

# Bootstrap procedure
for _ in range(num_bootstraps):
    # Generate a synthetic dataset using the estimated lambda
    synthetic_data = np.random.poisson(lam=lambda_hat, size=sum(observed_counts))
    
    # Bin the synthetic data to match the observed format (number of counts per 10-second interval)
    sythetic_hist, _ = np.histogram(synthetic_data, bins=np.arange(0, 19))
    
    # Calculate the new lambda estimate from the synthetic data
    new_lambda = np.average(click_tenseconds, weights=sythetic_hist)
    
    # Store the new lambda estimate
    bootstrap_lambdas.append(new_lambda)
    
# Calculate the sample variance of the observed data
observed_var = np.average((click_tenseconds-sample_mean)**2, weights=observed_counts)

# Calculate the CLT-based standard deviation for lambda estimates
clt_std = np.sqrt(observed_var / np.sum(observed_counts))

# Plot the distribution of bootstrap lambda estimates
plt.figure(figsize=(10, 6))
plt.hist(bootstrap_lambdas, bins=30, color='green', edgecolor='black', alpha=0.7,density=True,label='Bootstrap Distribution')

# Generate the normal distribution using the CLT-based approximation
x = np.linspace(min(bootstrap_lambdas), max(bootstrap_lambdas), 1000)
normal_approximation = norm.pdf(x, loc= sample_mean, scale=clt_std)

# Plot the normal distribution on top of the histogram
plt.plot(x, normal_approximation, 'r-', linewidth=2, label=f'Normal Approximation \n(mean={sample_mean:.2f}, std={clt_std:.2f})')

plt.title('Bootstrap Distribution of Lambda Estimates with CLT Normal Approximation', fontsize=14)
plt.xlabel('Lambda Estimate', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.grid(True)

plt.legend()

In [ ]:
# USE BOOTSRAP TO ESTIMATE DISTRIBUTION OF VARIANCE FOR NORMAL DISTRIBUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm


# Set random seed for reproducibility
np.random.seed(42)

# Generate data from a normal distribuion
mu = 10 # True mean of the normal distribution
sigma = 2 # True standard deviation of the normal distribution
N = 100 # Number of samples
data = np.random.normal(mu, sigma, N)

# Method of Moments estimates for mean and variance
estimated_mu = np.mean(data)
estimated_var = np.var(data,ddof=1)

# Number of synthetic datasets to generate
num_synthetic_samples = 1000
synthetic_variances = []

# Generate synthetic data using the estimated mean and variance and calculate variances
for _ in range(num_synthetic_samples):
    # Generate synthetic dataset from normal distribution using estimated parameters
    synthetic_data = np.random.normal(loc=estimated_mu, scale=np.sqrt(estimated_var), size=N)
    
    # Compute the sample variance for the synthetic dataset
    synthetic_variance = np.var(synthetic_data,ddof=1) # Unbiased estimate of variance
    synthetic_variances.append(synthetic_variance)
    
# Plot the distribution of the synthetic sample variances
plt.figure(figsize=(10, 6))
plt.hist(synthetic_variances, bins=30,color='blue',edgecolor='black',alpha=0.7,density=True,label='Synthetic Variance Distribution')

plt.title('Synthetic Distribution of Sample Variance With Normal Appx. and Chi-Square Fit', fontsize=14)
plt.xlabel('Sample Variance', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.grid(True)

plt.legend()

# Print the mean and confidence interval of the bootstrap lambda estimates
bootstrap_mean = np.mean(synthetic_variances)
bootstrap_std = np.std(synthetic_variances)
bootstrap_variance = np.var(synthetic_variances,ddof=1)
print(f"Mean of Bootstrap Sample Variance: {bootstrap_mean:.2f}")
print(f"Std Dev of Bootstrap Sample Variance: {bootstrap_std:.2f}")
print(f"Method of Moments Estimate of Variance: {bootstrap_variance:.2f}")

In [ ]:
# USE BOOTSTRAP TO ESTIMATE DISTRIBUTION OF VARIANCE FOR NORMAL DISTRIBUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, chi2

# Set random seed for reproducibility
np.random.seed(42)

# Generate data from a normal distribuion
mu = 10 # True mean of the normal distribution
sigma = 2 # True standard deviation of the normal distribution
N = 100 # Number of samples
data = np.random.normal(mu, sigma, N)

# Method of Moments estimates for mean and variance
estimated_mu = np.mean(data)
estimated_var = np.var(data,ddof=1)

# Number of synthetic datasets to generate
num_synthetic_samples = 1000
synthetic_variances = []

# Generate synthetic data using the estimated mean and variance and calculate variances
for _ in range(num_synthetic_samples):
    # Generate synthetic dataset from normal distribution using estimated parameters
    synthetic_data = np.random.normal(loc=estimated_mu, scale=np.sqrt(estimated_var), size=N)
    
    # Compute the sample variance for the synthetic dataset
    synthetic_variance = np.var(synthetic_data,ddof=1) # Unbiased estimate of variance
    synthetic_variances.append(synthetic_variance)
    
# Plot the distribution of the synthetic sample variances
plt.figure(figsize=(10, 6))
plt.hist(synthetic_variances, bins=30,color='blue',edgecolor='black',alpha=0.7,density=True,label='Synthetic Variance Distribution')

# Create a range of x values for the variance
x = np.linspace(min(synthetic_variances), max(synthetic_variances), 1000)

# Calculate the mean and std of the synthetic_variances list
sync_mean = np.mean(synthetic_variances)
sync_std  = np.std(synthetic_variances) 
sync_std = 0.3299 # I cannot figure out how to get the same "tallness" as the video (2026-4-7)

# 1. Normal Approximation (The "Yellow" Curve)
# Using sync_std ensures the curve height matches the histogram's density
pdf_normal = norm.pdf(x, loc=sync_mean, scale=sync_std)
plt.plot(x, pdf_normal, color='gold', linewidth=3, 
         label=f'Normal Approximation (CLT)\n(mean={sync_mean:.2f}, std={sync_std:.4f})')


# 2. Chi-Square Distribution (True Analytical Distribution)
# The sample variance S^2 is related to Chi-Square by: (N-1)*S^2 / sigma^2 ~ Chi2(N-1)
# Therefore, S^2 ~ (sigma^2 / (N-1)) * Chi2(N-1)
df = N - 1
# Scale factor to convert Chi-Square statistic back to Variance units
scale_factor = estimated_var / df 

# Convert variance x-values back to Chi-Square statistic values for the PDF calculation
x_chi2_stat = x / scale_factor
# Calculate the Chi-Square PDF
pdf_chi2_stat = chi2.pdf(x_chi2_stat, df=df)
# Scale the PDF to match the variance density (Jacobian adjustment: dx_stat = dx_var / scale_factor)
pdf_chi2 = pdf_chi2_stat / scale_factor

# Plot the Chi-Square distributions
plt.plot(x, pdf_chi2, 'cyan', linewidth=2, label=f'Chi-Square Fit (df={df})')

plt.title('Synthetic Distribution of Sample Variance With Normal Appx. and Chi-Square Fit', fontsize=14)
plt.xlabel('Sample Variance', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.ylim(0, 1.2)
plt.grid(True)

plt.legend()

# Print the mean and confidence interval of the bootstrap lambda estimates
print(f"Mean of Synthetic Sample Variance: {bootstrap_mean:.2f}")
print(f"Std Dev of Synthetic Sample Variance: {bootstrap_std:.2f}")
print(f"Method of Moments Estimate of Variance: {bootstrap_variance:.2f}")